# 真实数据全链路回测

用 tushare 真实 A 股数据跑通完整管道：
```
data → factors → strategies → portfolio → risk → backtest → visualization
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.data.fetcher import fetch_daily
from src.data.storage import save_parquet, load_parquet
from src.data.filters import detect_limit_price, detect_suspension
from src.data import validate_ohlcv
from src.strategies.mean_reversion import mean_reversion_signal
from src.risk.tradability import filter_tradable, enforce_t1
from src.portfolio.allocator import equal_weight
from src.risk.position_limit import apply_position_limit
from src.backtest.engine import BacktestEngine
from src.visualization.charts import plot_backtest_summary
from pathlib import Path

print('All imports OK')

## Step 1: 拉取真实数据

In [ ]:
STOCKS = ['000001', '600519', '000858']
START = '2025-05-01'
END = '2026-05-23'
RAW_DIR = Path('../data/raw')

frames = []
for code in STOCKS:
    path = RAW_DIR / f'{code}.parquet'
    if path.exists():
        df = load_parquet(path)
        print(f'{code}: loaded {len(df)} rows from cache')
    else:
        df = fetch_daily(code, START, END)
        save_parquet(df, path)
        print(f'{code}: fetched {len(df)} rows from tushare')
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(data)} rows, {data["code"].nunique()} stocks')
print(f'Date range: {data["date"].min().date()} ~ {data["date"].max().date()}')
data.head()

## Step 2: 数据校验与标注

In [ ]:
validate_ohlcv(data)
print('Schema validation passed')

data = detect_limit_price(data)
data = detect_suspension(data)

limit_up_count = data['limit_up'].sum()
limit_down_count = data['limit_down'].sum()
suspension_count = data['is_suspended'].sum()
print(f'Limit up: {limit_up_count}, Limit down: {limit_down_count}, Suspended: {suspension_count}')
data.tail()

## Step 3: 信号生成

In [ ]:
signals = mean_reversion_signal(data, window=20, num_std=2.0)

print(f'Total signals: {len(signals)}')
print(f'\nSignal distribution:')
print(signals['signal'].value_counts().sort_index())
print(f'\nBuy signals: {(signals["signal"] == 1).sum()}')
print(f'Sell signals: {(signals["signal"] == -1).sum()}')
print(f'Hold: {(signals["signal"] == 0).sum()}')

## Step 4: 风控过滤

In [ ]:
# 过滤涨跌停/停牌
filtered = filter_tradable(data, signals)
blocked = len(signals[signals['signal'] != 0]) - len(filtered[filtered['signal'] != 0])
print(f'Signals blocked by tradability filter: {blocked}')

# T+1 同日冲突处理
final_signals = enforce_t1(filtered)
print(f'\nFinal signal distribution:')
print(final_signals['signal'].value_counts().sort_index())

## Step 5: 组合分配

In [ ]:
CAPITAL = 1_000_000

# 等权分配
positions = equal_weight(final_signals, data[['date', 'code', 'close']], capital=CAPITAL)
print(f'Positions generated: {len(positions)} rows')
print(f'Dates with positions: {positions["date"].nunique()}')

# 仓位上限
positions = apply_position_limit(positions, max_weight=0.3)
print(f'After position limit: {len(positions)} rows')
positions.head(10)

## Step 6: 回测

In [ ]:
engine = BacktestEngine(capital=CAPITAL)
result = engine.run(positions, data[['date', 'code', 'close']])

print('=== Backtest Metrics ===')
for k, v in result['metrics'].items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print(f'\n=== Trade Summary ===')
print(f'Total trades: {len(result["trades"])}')
if not result['trades'].empty:
    print(f'\nTrades:')
    print(result['trades'].to_string(index=False))

print(f'\n=== Equity Curve ===')
print(f'Start: {result["equity_curve"].iloc[0]["equity"]:.2f}')
print(f'End: {result["equity_curve"].iloc[-1]["equity"]:.2f}')

## Step 7: 可视化

In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

fig = plot_backtest_summary(result)
fig.set_size_inches(14, 8)
fig.suptitle('Mean Reversion Strategy — 000001/600519/000858 (2025-05 ~ 2026-05)', fontsize=14)
fig.tight_layout()

out_path = output_dir / 'real_backtest_summary.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Chart saved to {out_path}')
plt.close(fig)

## Step 8: 结果分析

（运行后填写观察）